In [1]:
from pyspark.sql import SparkSession

In [109]:
spark = SparkSession.builder.master("local[6]").appName("Local Spark") .config('spark.ui.port', '4040') .getOrCreate()
sc = spark.sparkContext

sc


'3.5.0'

In [13]:
# Using a small sized part of the dataset, for now
dataset_path = "/home/jovyan/work/dataset/output/csv/part-00172-0deb703d-e274-494e-ae41-e0f3a8f731b9-c000.csv"

rddTelemetry = sc.textFile(dataset_path)
print(f"Number of partitions: {rddTelemetry.getNumPartitions()}")


Number of partitions: 2


In [146]:
def parseTelemetryRow(row):
    splitted = row.split(",")

    (year, event, session_type, driver_name, lap_number, distance_driver_ahead, driver_ahead, acc_x, acc_y,
     acc_z, brake, distance, drs, gear, rel_distance, rpm, speed, throttle, time, x, y, z) = [x for x in splitted]

    return ((event, session_type, driver_name), (float(acc_y), int(brake), float(rpm), float(speed), float(throttle), rel_distance))

In [174]:
# parsing each row by creating key-value rows
columnNames = rddTelemetry.take(1)
print(columnNames)
rddTelemetryKV = (rddTelemetry \
    .filter(lambda x: x != columnNames[0])
    .map(lambda x: parseTelemetryRow(x))
    .filter(lambda x: x[1][5] != 'None'))

rddTelemetryKV = rddTelemetryKV.map(lambda x: (x[0], (x[1][0],x[1][1],x[1][2],x[1][3],x[1][4])))


['year,event,sessionType,driverName,lapNumber,DistanceToDriverAhead,DriverAhead,acc_x,acc_y,acc_z,brake,distance,drs,gear,rel_distance,rpm,speed,throttle,time,x,y,z']


In [175]:
print(f"Number of rows: {rddTelemetryKV.count()}")

Number of rows: 44190


In [179]:
# aggregating by key based on 95th percentile for lateral acceleration, average on braking, engine rpms while accelerating

# sequencing function on (acc_y, brake, rpm, speed, throttle) to calc average and give as a result (max_acc_y, sum_brake, sum_rpm, rows_throttle_speed_threshold, total_rows)
seqFunc = (
    lambda x, y:
    (x[0] if x[0] > y[0] else y[0],
     x[1] + y[1],
     (x[2] + y[2]) if (y[3] > 120 and y[4] > 35) else x[2],
     (x[3] + 1) if (y[3] > 120 and y[4] > 35) else x[3],
     x[4] + 1)
)

#combining function between partitions
combFunc = (
    lambda x, y:
    (x[0] if x[0] > y[0] else y[0],
     x[1] + y[1],
     x[2] + y[2],
     x[3] + y[3],
     x[4] + y[4])
)

#accumulator is (max_acc_y, sum_brake, sum_rpm, rows_throttle_speed_threshold, total_rows)
rddTelemetryKV\
.aggregateByKey((-30.0, 0, 0, 0, 0), seqFunc, combFunc)\
    .mapValues(lambda x: (x[0], x[1]/x[4], x[2]/x[3]))\
    .sortByKey()\
    .collect()

# (rddTelemetryKV\
#     .map(lambda x: (x[0], (x[1][1], 1)))\
#     .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))\
#     .mapValues(lambda x: x[0]/x[1])\
#     .collect())

[(('Abu Dhabi Grand Prix', 'Practice 3', 'HAD'),
  (38.351838288949736, 0.08443908323281062, 10947.580195295099)),
 (('Australian Grand Prix', 'Qualifying', 'LEC'),
  (28.156959137841486, 0.19047619047619047, 10335.765102031572)),
 (('Austrian Grand Prix', 'Qualifying', 'HUL'),
  (25.598682353456834, 0.4239766081871345, 9942.482813999592)),
 (('Azerbaijan Grand Prix', 'Practice 1', 'COL'),
  (33.95125058292253, 0.07566024268379729, 11004.838253729573)),
 (('Azerbaijan Grand Prix', 'Practice 2', 'HAD'),
  (30.07428169282815, 0.13906359189378056, 10802.60423539824)),
 (('Azerbaijan Grand Prix', 'Practice 2', 'HAM'),
  (37.2836255044735, 0.226546212647672, 11031.513339106643)),
 (('Azerbaijan Grand Prix', 'Practice 2', 'SAI'),
  (23.228428166932957, 0.1796657381615599, 10622.54122606894)),
 (('Azerbaijan Grand Prix', 'Practice 3', 'ANT'),
  (32.17393972733111, 0.1339031339031339, 10545.109064642851)),
 (('Azerbaijan Grand Prix', 'Race', 'HAD'),
  (44.78315881584814, 0.2148997134670487, 11